In [1]:
import spacy
from spacy.tokens import Doc, DocBin
from sklearn.model_selection import train_test_split
print(spacy.__version__)
import pandas as pd
from tqdm import tqdm

3.8.11


In [2]:
import os
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
os.chdir("/content/drive/MyDrive/OICE-POS")

In [4]:
os.listdir()

['output',
 'dl_xml_utd_corpus.parquet',
 'dev.spacy',
 'test.spacy',
 'train.spacy',
 'config.cfg',
 'OICE_LePOS_Training.ipynb',
 'OICE_LePOS_Examples.ipynb',
 'OICE_LePOS_Eval.ipynb',
 'oice-lepos-spacy-ud']

In [5]:
from spacy.training import Example

# 1 Load best model
nlp = spacy.load("oice-lepos-spacy-ud")

# 2 Load the gold test set
docbin = DocBin().from_disk("test.spacy")
gold_docs = list(docbin.get_docs(nlp.vocab))

# 3 Build Example objects
examples = []
for gold in tqdm(gold_docs):
    # Predict on the same text
    pred = nlp(gold.text)
    # Create an Example pairing the predicted doc with the gold doc
    example = Example(pred, gold)
    examples.append(example)

# 4 Evaluate
results = nlp.evaluate(examples)

100%|██████████| 2707/2707 [00:46<00:00, 58.64it/s]


In [6]:
# 5 Metrics
print(f"Tag accuracy (POS):  {results['tag_acc']:.4f}")
print(f"Lemma accuracy: {results['lemma_acc']:.4f}")
print("\nFull results:\n", results)

Tag accuracy (POS):  0.9603
Lemma accuracy: 0.9328

Full results:
 {'token_acc': None, 'token_p': None, 'token_r': None, 'token_f': None, 'tag_acc': 0.9603103551178752, 'lemma_acc': 0.9327907811725283, 'speed': 2086.454426286412}


In [7]:
from sklearn.metrics import classification_report

y_true, y_pred = [], []

for ex in tqdm(examples, desc="POS tags"):
    ref_tokens = ex.reference
    pred_tokens = ex.predicted

    # Align tokens by index; skip misaligned Docs
    if len(ref_tokens) != len(pred_tokens):
        continue

    y_true.extend([t.tag_ for t in ref_tokens])
    y_pred.extend([t.tag_ for t in pred_tokens])

print("\nDetailed POS classification report:")
print(classification_report(y_true, y_pred))

POS tags: 100%|██████████| 2707/2707 [00:00<00:00, 126361.73it/s]


Detailed POS classification report:



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

                   0.00      0.00      0.00        53
         ADJ       0.92      0.92      0.92      1417
         ADP       0.96      0.96      0.96      2089
         ADV       0.98      0.98      0.98      2360
       CCONJ       0.96      0.96      0.96      2478
         DET       0.98      0.98      0.98      1920
        NOUN       0.96      0.95      0.96      4305
         NUM       1.00      1.00      1.00       278
        PART       0.83      0.90      0.86       915
        PRON       0.99      1.00      1.00      2328
       PROPN       0.98      0.99      0.98      1411
       PUNCT       1.00      1.00      1.00      2324
       SCONJ       0.73      0.58      0.64       142
        VERB       0.97      0.98      0.97      6008

    accuracy                           0.97     28028
   macro avg       0.88      0.87      0.87     28028
weighted avg       0.96      0.97      0.97     28028



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
y_true, y_pred = [], []

for ex in tqdm(examples, desc="Collecting aligned POS tags"):
    ref_tokens = ex.reference
    pred_tokens = ex.predicted

    # Align tokens by index; skip misaligned Docs
    if len(ref_tokens) != len(pred_tokens):
        continue

    y_true.extend([t.lemma_ for t in ref_tokens])
    y_pred.extend([t.lemma_ for t in pred_tokens])

print("\nDetailed POS classification report:")
print(classification_report(y_true, y_pred))


Detailed POS classification report:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

                     precision    recall  f1-score   support

                          0.00      0.00      0.00      2377
                  ,       0.00      0.00      0.00         0
                 ,l       0.00      0.00      0.00         0
                 -a       1.00      1.00      1.00         1
                  .       0.00      0.00      0.00         0
                 .l       0.00      0.00      0.00         0
                 10       0.67      1.00      0.80         2
                100       1.00      1.00      1.00         1
                 12       1.00      1.00      1.00         9
                 15       1.00      1.00      1.00         3
                 16       0.00      0.00      0.00         1
                  2       0.86      0.86      0.86         7
                 20       1.00      1.00      1.00         1
                  3       0.40      0.33      0.36         6
                 30       1.00      1.00      1.00         1
                  4    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
